# 7장. 그래프로 데이터의 이야기를 보여주기

이 노트북의 목표는 그래프를 많이 만드는 것이 아닙니다. **분석 질문에 맞는 그래프를 선택하고, 그래프에 들어간 데이터 범위와 수치를 검증하며, 그래프가 보여 주는 사실과 보여 주지 못하는 것을 구분하는 것**이 핵심입니다.

금액성 그래프는 기본적으로 `order_status == "completed"` 범위를 사용합니다. `line_total`은 주문 상세 한 행의 `quantity × unit_price`이며 회계상 순매출로 단정하지 않습니다.

## 0. 사용 방법

- 위에서부터 차례대로 실행합니다.
- 입력은 Chapter 05에서 만든 `data/processed/*_clean.csv`입니다.
- 파일이 없다면 프로젝트 루트에서 `python scripts/preprocess_data.py`를 먼저 실행합니다.
- 그래프를 보기 전에 **시각화용 집계 검증표**를 먼저 확인합니다.
- 대표 그래프마다 `관찰 → 해석 → 한계 → 다음 질문`을 Markdown 셀에 직접 작성합니다.
- 실제 실행 결과 숫자를 보고 작성하며 강의안의 예시를 정답처럼 사용하지 않습니다.

## 1. 질문에 맞는 그래프를 먼저 고릅니다

| 분석 질문 | 그래프 | 데이터 범위 |
| --- | --- | --- |
| 카테고리별 완료 주문 금액은 어떻게 다른가? | 막대그래프 | completed 주문 상세 |
| 월별 완료 주문 금액은 어떻게 변하는가? | 선그래프 | completed + 유효한 주문일 |
| 상품 가격은 어느 구간에 몰려 있는가? | 히스토그램 | 전체 상품 마스터 |
| 상품 가격과 완료 주문 판매 수량은 어떤 패턴인가? | 산점도 | completed 상품 집계 |
| 완료 주문 구매 금액 상위 고객군은 어떻게 구성되는가? | 가로 막대그래프 | completed 고객 집계 |
| 주문 상태별 주문 수는 어떻게 다른가? | 막대그래프 | 전체 orders |

> 그래프 종류보다 먼저 **질문과 데이터 범위**를 고정합니다.

In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
SEARCH_DIRS = [CURRENT_DIR, *CURRENT_DIR.parents]

NOTEBOOK_DIR = next(
    (path for path in SEARCH_DIRS if (path / "chapter07.ipynb").exists()),
    None,
)
if NOTEBOOK_DIR is None:
    NOTEBOOK_DIR = next(
        (path / "chapter07" for path in SEARCH_DIRS
         if (path / "chapter07" / "chapter07.ipynb").exists()),
        CURRENT_DIR,
    )

PROJECT_CANDIDATES = []
for path in SEARCH_DIRS:
    PROJECT_CANDIDATES.append(path)
    nested_course = path / "llm-data-analysis-course"
    if nested_course.is_dir():
        PROJECT_CANDIDATES.append(nested_course)

PROJECT_ROOT = next(
    (path for path in PROJECT_CANDIDATES
     if (path / "src" / "visualization.py").exists()),
    CURRENT_DIR,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_CANDIDATES = [
    NOTEBOOK_DIR / "processed",
    NOTEBOOK_DIR / "data" / "processed",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "practice" / "chapter05" / "data" / "processed",
]
PROCESSED_DIR = next(
    (path for path in DATA_CANDIDATES
     if all((path / name).exists() for name in [
         "customers_clean.csv", "products_clean.csv",
         "orders_clean.csv", "order_items_clean.csv",
     ])),
    None,
)
if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "전처리 CSV를 찾지 못했습니다. Chapter 05에서 생성한 clean CSV를 "
        "chapter07/processed 또는 프로젝트 data/processed에 준비하세요."
    )

REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
IMAGE_DIR = NOTEBOOK_DIR / "images"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("현재 실행 위치:", CURRENT_DIR)
print("Notebook 폴더:", NOTEBOOK_DIR)
print("프로젝트 루트:", PROJECT_ROOT)
print("전처리 데이터 폴더:", PROCESSED_DIR)
print("그래프 저장 폴더:", FIGURE_DIR)
print("Evidence 이미지 폴더:", IMAGE_DIR)


## 2. 검증된 시각화용 데이터를 준비합니다

공통 모듈은 그래프를 그리기 전에 다음을 검사합니다.

- 필수 컬럼
- PK 결측·중복
- `many_to_one` 병합
- 주문/상품 미매칭
- 숫자형 변환 실패
- `completed` 주문 범위
- 카테고리·상품·고객·월별 집계 총합

문제가 있으면 그래프를 만들기 전에 중단합니다.

In [ ]:
# 입력 CSV는 읽기만 합니다. 아래 전처리·집계는 메모리에서 수행하며
# data/processed의 원본 CSV에는 저장하거나 덮어쓰지 않습니다.
import pandas as pd
import numpy as np

customers = pd.read_csv(PROCESSED_DIR / "customers_clean.csv", encoding="utf-8-sig")
products = pd.read_csv(PROCESSED_DIR / "products_clean.csv", encoding="utf-8-sig")
orders = pd.read_csv(PROCESSED_DIR / "orders_clean.csv", encoding="utf-8-sig")
order_items = pd.read_csv(PROCESSED_DIR / "order_items_clean.csv", encoding="utf-8-sig")

required_columns = {
    "customers": {"customer_id"},
    "products": {"product_id", "product_name", "category", "price"},
    "orders": {"order_id", "customer_id", "order_date", "order_status"},
    "order_items": {
        "order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_total"
    },
}
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

for name, required in required_columns.items():
    missing_columns = required - set(datasets[name].columns)
    if missing_columns:
        raise ValueError(f"{name} 필수 컬럼 누락: {sorted(missing_columns)}")

pk_audit = []
for name, key in primary_keys.items():
    frame = datasets[name]
    pk_audit.append({
        "dataset": name,
        "rows": len(frame),
        "pk_null_count": int(frame[key].isna().sum()),
        "pk_duplicate_count": int(frame[key].duplicated().sum()),
    })
pk_audit = pd.DataFrame(pk_audit)
if pk_audit[["pk_null_count", "pk_duplicate_count"]].to_numpy().sum() > 0:
    raise ValueError("PK 결측 또는 중복이 있어 집계를 중단합니다.")

for column in ["quantity", "unit_price", "line_total"]:
    order_items[column] = pd.to_numeric(order_items[column], errors="coerce")
numeric_failures = int(order_items[["quantity", "unit_price", "line_total"]].isna().sum().sum())
if numeric_failures:
    raise ValueError(f"주문 상세 숫자 변환 실패: {numeric_failures}개")

line_total_mismatch = ~np.isclose(
    order_items["line_total"],
    order_items["quantity"] * order_items["unit_price"],
    rtol=0,
    atol=1e-8,
)
line_total_mismatch_count = int(line_total_mismatch.sum())

# 주문 FK가 없는 2개 상세행은 상태·날짜를 알 수 없으므로 주문 분석에서 제외합니다.
# 그 행 자체는 원본에 그대로 두고, 건수와 총액만 관계 감사에 기록합니다.
unmatched_orders = order_items.loc[
    ~order_items["order_id"].isin(orders["order_id"])
].copy()
items_with_orders = order_items.merge(
    orders,
    on="order_id",
    how="inner",
    validate="many_to_one",
)
completed_order_sales = items_with_orders.loc[
    items_with_orders["order_status"].eq("completed")
].copy()
completed_amount = float(completed_order_sales["line_total"].sum())
completed_order_count = int(completed_order_sales["order_id"].nunique())

# 상품 FK 누락 상세는 완료 금액을 보존해 카테고리 합계의
# '상품 정보 없음' 범주에 넣고, 상품 가격·수량 산점도에서만 제외합니다.
completed_with_products = completed_order_sales.merge(
    products[["product_id", "product_name", "category", "price"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_match",
)
missing_product_mask = completed_with_products["product_match"].eq("left_only")
missing_product_items = completed_with_products.loc[missing_product_mask].copy()
completed_with_products["category"] = completed_with_products["category"].fillna(
    "상품 정보 없음"
)

category_sales = (
    completed_with_products.groupby("category", as_index=False, dropna=False)
    .agg(
        total_quantity=("quantity", "sum"),
        completed_amount=("line_total", "sum"),
    )
    .sort_values("completed_amount", ascending=False)
    .reset_index(drop=True)
)
category_sales["amount_ratio_pct"] = (
    category_sales["completed_amount"] / completed_amount * 100
).round(2)

# 월별 집계에서는 주문일 결측 상세만 제외하고, 월별 주문 수는 고유 order_id로 셉니다.
completed_order_sales["order_date_parsed"] = pd.to_datetime(
    completed_order_sales["order_date"], errors="coerce"
)
invalid_date_sales = completed_order_sales.loc[
    completed_order_sales["order_date_parsed"].isna()
].copy()
valid_date_sales = completed_order_sales.loc[
    completed_order_sales["order_date_parsed"].notna()
].copy()
valid_date_sales["order_month"] = (
    valid_date_sales["order_date_parsed"].dt.to_period("M").astype(str)
)
monthly_sales = (
    valid_date_sales.groupby("order_month", as_index=False)
    .agg(
        completed_amount=("line_total", "sum"),
        completed_order_count=("order_id", "nunique"),
    )
    .sort_values("order_month")
    .reset_index(drop=True)
)
monthly_sales["average_completed_order_amount"] = (
    monthly_sales["completed_amount"] / monthly_sales["completed_order_count"]
).round(0)

# 산점도는 상품 정보가 연결된 완료 주문만 사용합니다.
linked_product_sales = completed_with_products.loc[~missing_product_mask].copy()
product_sales = (
    linked_product_sales.groupby(
        ["product_id", "product_name", "category", "price"], as_index=False
    )
    .agg(
        completed_quantity=("quantity", "sum"),
        completed_amount=("line_total", "sum"),
    )
    .sort_values("completed_amount", ascending=False)
    .reset_index(drop=True)
)

# 고객 ID는 내부 집계에만 쓰고, 그래프에는 금액 순위 별칭만 전달합니다.
customer_sales_internal = (
    completed_order_sales.groupby("customer_id", as_index=False, dropna=False)
    .agg(
        completed_order_count=("order_id", "nunique"),
        completed_amount=("line_total", "sum"),
    )
    .sort_values(["completed_amount", "customer_id"], ascending=[False, True])
    .reset_index(drop=True)
)
customer_sales_internal["average_completed_order_amount"] = (
    customer_sales_internal["completed_amount"]
    / customer_sales_internal["completed_order_count"]
).round(0)
customer_sales_internal.insert(
    0, "customer_rank", range(1, len(customer_sales_internal) + 1)
)
customer_sales = customer_sales_internal.drop(columns="customer_id").copy()
customer_sales["customer_id"] = customer_sales["customer_rank"].map(
    lambda rank: f"고객 순위 {rank:03d}"
)
customer_sales_public = customer_sales[
    [
        "customer_rank",
        "completed_order_count",
        "completed_amount",
        "average_completed_order_amount",
    ]
].copy()

# 주문에 적힌 고객 키가 customers 마스터에 없더라도 주문 금액은 보존합니다.
customer_profile_check = completed_order_sales.merge(
    customers[["customer_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="customer_match",
)
missing_customer_profile = customer_profile_check["customer_match"].eq("left_only")
missing_customer_profile_amount = float(
    customer_profile_check.loc[missing_customer_profile, "line_total"].sum()
)

order_status = (
    orders.groupby("order_status", as_index=False)
    .agg(order_count=("order_id", "nunique"))
    .sort_values("order_status")
    .reset_index(drop=True)
)

product_linked_amount = float(linked_product_sales["line_total"].sum())
monthly_valid_amount = float(valid_date_sales["line_total"].sum())
validation_rows = [
    ("line_total_equals_quantity_times_unit_price", 0.0, float(line_total_mismatch_count)),
    ("category_amount_equals_completed_amount", completed_amount, float(category_sales["completed_amount"].sum())),
    ("customer_amount_equals_completed_amount", completed_amount, float(customer_sales["completed_amount"].sum())),
    ("product_group_amount_equals_product_linked_amount", product_linked_amount, float(product_sales["completed_amount"].sum())),
    ("monthly_amount_equals_valid_date_amount", monthly_valid_amount, float(monthly_sales["completed_amount"].sum())),
]
visualization_validation = pd.DataFrame(
    [
        {
            "check": name,
            "expected": expected,
            "actual": actual,
            "difference": actual - expected,
            "passed": bool(np.isclose(actual, expected, rtol=0, atol=1e-8)),
        }
        for name, expected, actual in validation_rows
    ]
)

relationship_audit = pd.DataFrame([
    {
        "check": "order_items.order_id exists in orders.order_id",
        "unmatched_rows": len(unmatched_orders),
        "affected_amount": float(unmatched_orders["line_total"].sum()),
        "treatment": "주문 상태·날짜를 알 수 없어 주문 기반 집계에서 제외; 원본 CSV 유지",
    },
    {
        "check": "completed order_items.product_id exists in products.product_id",
        "unmatched_rows": len(missing_product_items),
        "affected_amount": float(missing_product_items["line_total"].sum()),
        "treatment": "카테고리에서는 상품 정보 없음으로 보존; 상품별 산점도에서 제외",
    },
    {
        "check": "completed orders.customer_id exists in customers.customer_id",
        "unmatched_rows": int(missing_customer_profile.sum()),
        "affected_amount": missing_customer_profile_amount,
        "treatment": "금액 합계는 보존; 고객 프로필 해석은 하지 않음",
    },
    {
        "check": "completed order_date is valid",
        "unmatched_rows": len(invalid_date_sales),
        "affected_amount": float(invalid_date_sales["line_total"].sum()),
        "treatment": "월별 집계에서만 제외; 다른 완료 금액 집계에는 보존",
    },
])

# 차트 공통 함수에 필요한 명시적 집계 결과를 제공합니다.
viz_data = {
    "products": products,
    "orders": orders,
    "completed_order_sales": completed_order_sales,
    "category_sales": category_sales,
    "monthly_sales": monthly_sales,
    "product_sales": product_sales,
    "customer_sales": customer_sales,
    "order_status": order_status,
    "invalid_date_sales": invalid_date_sales,
    "relationship_audit": relationship_audit,
    "visualization_validation": visualization_validation,
}

print("입력 데이터 행 수:", {name: len(frame) for name, frame in datasets.items()})
print("완료 주문 상세:", len(completed_order_sales), "행 /", completed_order_count, "개 주문")
print("완료 주문 상세 금액 합계:", f"{completed_amount:,.0f}원")
print("상품 FK 미매칭 완료 상세:", len(missing_product_items), "행; 금액은 카테고리 합계에 보존")
print("주문일 결측 완료 상세:", len(invalid_date_sales), "행; 월별 집계에서만 제외")


In [ ]:
import shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from src.visualization import (
    setup_korean_font,
    plot_category_sales,
    plot_monthly_sales,
    plot_product_price_hist,
    plot_price_quantity_scatter,
    plot_top_customers,
    plot_order_status,
    create_all_figures,
    create_visualization_summary,
    build_visualization_report,
)

# Codespaces에서는 설치된 NanumGothic 파일을 직접 등록합니다.
font_candidates = [
    Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
    Path("/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"),
    Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
]
font_path = next((path for path in font_candidates if path.exists()), None)
if font_path is not None:
    fm.fontManager.addfont(str(font_path))
    korean_font = fm.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams["font.family"] = korean_font
    plt.rcParams["axes.unicode_minus"] = False
    selected_font = korean_font
else:
    selected_font = setup_korean_font()
print("사용 한글 폰트:", selected_font or "자동 선택 실패 — 설치된 한글 폰트 확인 필요")

# 관측이 없는 월은 0으로 채우지 않고 NaN으로 두어 선이 공백을 가로질러
# 추세처럼 이어지지 않도록 그래프 전용 표를 만듭니다.
all_months = pd.period_range(
    monthly_sales["order_month"].min(), monthly_sales["order_month"].max(), freq="M"
).astype(str)
monthly_sales_for_plot = (
    monthly_sales.set_index("order_month")
    .reindex(all_months)
    .rename_axis("order_month")
    .reset_index()
)
viz_data_for_figures = dict(viz_data)
viz_data_for_figures["monthly_sales"] = monthly_sales_for_plot

print("주문 FK / 상품 FK / 고객 프로필 FK / 날짜 결측 점검")
display(relationship_audit)


In [ ]:
viz_data["visualization_validation"]

### 검증 결과 기록

첨부된 Chapter 05 clean CSV를 기준으로 확인했습니다. 주문 상세 752행 중 orders에 연결되지 않은 2행(합계 922,000원)은 주문 상태를 알 수 없어 주문 분석에서 제외했습니다. 연결된 완료 주문 상세는 446행·179개 주문이며 금액 합계는 141,340,000원입니다.

상품 FK가 없는 완료 상세 16행·4,015,000원은 카테고리 집계에서 `상품 정보 없음`으로 보존합니다. 상품 정보가 필요한 산점도에서는 제외합니다. 날짜 결측 완료 상세 6행·1,903,000원은 월별 집계에서만 제외하고 다른 완료 금액 집계에는 남깁니다. 고객 프로필 FK가 없는 완료 상세 1행·376,000원도 금액 집계에는 보존합니다.

집계 코드가 `passed=True`를 출력하는지 실행 후 확인해야 합니다. 이 노트북의 집계 코드는 원본 CSV를 수정하지 않습니다.

## 3. 카테고리별 완료 주문 금액 — 막대그래프

범주별 크기 비교이므로 막대그래프를 사용합니다. 값 축은 0에서 시작합니다. 금액이 크다는 사실만으로 선호도가 높다고 단정하지 않습니다.

In [ ]:
from src.visualization import plot_category_sales

category_sales[["category", "total_quantity", "completed_amount", "amount_ratio_pct"]]

In [ ]:
category_path = plot_category_sales(
    category_sales,
    FIGURE_DIR / "ch07_category_completed_amount_bar.png",
    show=True,
)
shutil.copy2(category_path, IMAGE_DIR / "graph01.png")
category_path

### 그래프 1 기록

- **관찰:** 카테고리별 완료 금액 합계는 141,340,000원입니다. 패션 32,940,000원, 식품 25,035,000원, 전자기기 21,485,000원 순입니다. 상품 정보가 없는 16행의 4,015,000원은 별도 범주로 표시됩니다.
- **해석:** 현재 데이터에서 관측된 카테고리별 완료 주문 금액 규모를 비교할 수 있습니다. 띄어쓰기 차이가 있는 카테고리는 원문 확인 전까지 별도 범주로 둡니다.
- **한계:** 금액만으로 선호도·이익·차이의 원인을 알 수 없습니다. 상품 정보가 없는 행은 특정 카테고리로 배정할 근거가 없습니다.
- **다음 질문:** 표기 정합성을 확인한 뒤 카테고리별 수량·상품 수·평균 단가를 비교하면 금액 차이는 어떻게 달라지는가?

## 4. 월별 완료 주문 금액 — 선그래프

시간 순서가 핵심인 질문이므로 선그래프를 사용합니다. 첫 달과 마지막 달이 완전한 한 달인지, 중간 월이 비어 있다면 실제 0인지 데이터 누락인지 확인합니다.

In [ ]:
from src.visualization import plot_monthly_sales

monthly_sales[
    [
        "order_month",
        "completed_amount",
        "completed_order_count",
        "average_completed_order_amount",
    ]
]

In [ ]:
monthly_path = plot_monthly_sales(
    monthly_sales_for_plot,
    FIGURE_DIR / "ch07_monthly_completed_amount_line.png",
    show=True,
)
shutil.copy2(monthly_path, IMAGE_DIR / "graph02.png")
monthly_path

### 그래프 2 기록

- **관찰:** 유효한 주문일의 월별 완료 금액 합계는 139,437,000원입니다. 관측 월 중 2026년 8월이 16,980,000원·15개 주문으로 가장 높고, 2027년 1월은 909,000원·1개 주문입니다. 날짜 결측 6행·1,903,000원은 월별 집계에서 제외했습니다.
- **해석:** 월별 관측 금액과 주문 수가 함께 달라집니다. 2026년 9~12월은 집계 행이 없으며 그래프에서는 해당 기간에 선이 끊기도록 표시합니다.
- **한계:** 집계 행이 없는 달이 실제 0건인지 자료 누락인지는 알 수 없습니다. 첫 달과 마지막 달이 완전한 기간인지도 확인이 필요합니다. 프로모션·계절성 등 원인을 단정하지 않습니다.
- **다음 질문:** 누락 월의 원본 자료와 월별 주문 수·평균 주문 금액을 함께 확인하면 어떤 차이가 설명되는가?

## 5. 상품 가격 분포 — 히스토그램

상품 가격은 상품 마스터의 속성이므로 주문 상태와 무관하게 전체 상품을 봅니다. `bins` 값에 따라 모양이 달라질 수 있으므로 구간 수 하나만 보고 이상값을 확정하지 않습니다.

In [ ]:
from src.visualization import plot_product_price_hist

products["price"].describe()

In [ ]:
price_hist_path = plot_product_price_hist(
    products,
    FIGURE_DIR / "ch07_product_price_hist.png",
    show=True,
    bins=20,
)
price_hist_path

### bins 비교 실습

- **공통 패턴:** 상품 가격은 96개이며 중앙값은 94,500원, 최댓값은 9,999,999원입니다. bins=10·20·30 모두 극단적으로 큰 최댓값이 전체 범위를 넓혀 일반 가격대가 왼쪽에 몰려 보이는 현상이 유지됩니다.
- **설정에 따른 차이:** bins가 바뀌면 구간 폭과 경계가 달라집니다. 이 데이터에서는 구간 수를 늘려도 일반 가격대의 세부 분포가 충분히 드러나지 않습니다.
- **판단:** 최댓값을 오류로 단정하지 않습니다. 원본 상품 정보를 확인하고, 필요하면 축 범위를 달리한 보조 그래프를 전체 범위 그래프와 함께 비교합니다.

In [ ]:
from src.visualization import plot_product_price_hist

bin_comparison_paths = []
for bins in (10, 20, 30):
    output_path = FIGURE_DIR / f"ch07_product_price_hist_bins_{bins}.png"
    path = plot_product_price_hist(
        products,
        output_path,
        show=True,
        bins=bins,
    )
    bin_comparison_paths.append(path)
    print(f"bins={bins}:", path)

## 6. 상품 가격과 완료 주문 판매 수량 — 산점도

한 점은 상품 하나입니다. 관계가 보여도 **가격이 판매 수량의 원인**이라고 단정하지 않습니다.

In [ ]:
from src.visualization import plot_price_quantity_scatter

product_sales[["price", "completed_quantity", "completed_amount"]].head(10)

In [ ]:
scatter_path = plot_price_quantity_scatter(
    product_sales,
    FIGURE_DIR / "ch07_price_completed_quantity_scatter.png",
    show=True,
)
shutil.copy2(scatter_path, IMAGE_DIR / "graph03.png")
scatter_path

### 그래프 3 기록

- **관찰:** 산점도는 상품 정보가 연결된 완료 주문만 사용합니다. 연결된 상품별 완료 금액 합계는 137,325,000원입니다. 가격 최댓값 9,999,999원의 상품은 완료 판매 수량 8개입니다. 대부분의 상품은 상대적으로 낮은 가격대에 모입니다.
- **해석:** 가격과 판매 수량 사이의 뚜렷한 단순 선형 패턴은 확인하기 어렵고, 극단 가격이 나머지 점을 왼쪽에 압축해 보이게 합니다.
- **한계:** 상품 FK가 없는 16행은 상품 가격을 알 수 없어 산점도에서 제외했습니다. 카테고리·할인·재고·노출·판매 가능 기간을 통제하지 않아 인과관계를 말할 수 없습니다.
- **다음 질문:** 극단 가격의 원문을 확인하고 카테고리·판매 가능 기간별로 나누면 가격과 판매 수량의 관계가 어떻게 달라지는가?

## 7. 완료 주문 구매 금액 상위 고객 — 익명 가로 막대그래프

공개 그래프에서는 고객 이름이나 원본 고객 ID를 직접 표시하지 않고 **순위 기반 익명 라벨**을 사용합니다. Top 10은 전체 고객 분포가 아니라 일부 고객만 보여 줍니다.

In [ ]:
# 실제 고객 ID를 출력하지 않고 순위와 집계 지표만 확인합니다.
customer_sales_public.head(10)

In [ ]:
top_customer_path = plot_top_customers(
    customer_sales,
    FIGURE_DIR / "ch07_top_customers_anonymized_barh.png",
    show=True,
    top_n=10,
)
top_customer_path

### 해석 주의

구매 금액이 높다고 바로 '충성 고객'이라고 부르지 않습니다.

- 반복 주문 횟수는 많은가?
- 한 번의 큰 주문 때문인가?
- 평균 주문 금액은 어떤가?
- 분석 기간이 충분한가?

## 8. 전체 주문 상태 — 막대그래프

이 그래프만 전체 주문을 사용합니다. 상태별 규모는 보여 주지만 취소·환불의 원인을 설명하지 않습니다.

In [ ]:
from src.visualization import plot_order_status

order_status

In [ ]:
order_status_path = plot_order_status(
    order_status,
    FIGURE_DIR / "ch07_order_status_bar.png",
    show=True,
)
order_status_path

## 9. 그래프와 숫자를 교차 검증합니다

대표 그래프 3개 이상에 대해 다음을 확인합니다.

```text
그래프에 표시된 값
=
그래프 원본 집계표의 값
```

또한 다음을 점검합니다.

- 축 이름과 단위가 맞는가?
- 범주 정렬과 날짜 정렬이 맞는가?
- 막대그래프 값 축이 불필요하게 잘리지 않았는가?
- 제목이 실제 분석 범위를 반영하는가?
- 데이터가 없는 구간을 임의로 0으로 만들지 않았는가?

In [ ]:
# 그래프에 전달한 집계표의 금액·건수와 기준 합계를 자동 비교합니다.
graph_value_checks = pd.DataFrame([
    {
        "graph": "카테고리별 완료 금액",
        "source_total": completed_amount,
        "graph_data_total": float(category_sales["completed_amount"].sum()),
    },
    {
        "graph": "유효 주문일 월별 완료 금액",
        "source_total": monthly_valid_amount,
        "graph_data_total": float(monthly_sales["completed_amount"].sum()),
    },
    {
        "graph": "상품 가격-완료 판매 수량 산점도",
        "source_total": product_linked_amount,
        "graph_data_total": float(product_sales["completed_amount"].sum()),
    },
    {
        "graph": "익명 고객 상위 금액",
        "source_total": completed_amount,
        "graph_data_total": float(customer_sales["completed_amount"].sum()),
    },
    {
        "graph": "주문 상태별 주문 수",
        "source_total": len(orders),
        "graph_data_total": float(order_status["order_count"].sum()),
    },
])
graph_value_checks["difference"] = (
    graph_value_checks["graph_data_total"] - graph_value_checks["source_total"]
)
graph_value_checks["passed"] = np.isclose(
    graph_value_checks["source_total"],
    graph_value_checks["graph_data_total"],
    rtol=0,
    atol=1e-8,
)
display(graph_value_checks)
assert graph_value_checks["passed"].all(), "그래프 원본 집계값 검증 실패"
print("그래프 집계값 교차 검증: PASS")

## 10. 공통 함수로 전체 그래프와 보고서를 다시 생성합니다

Notebook에서 개별 그래프를 이해한 뒤 공통 함수를 사용해 같은 결과를 재현합니다.

In [ ]:
saved_figures = create_all_figures(viz_data_for_figures, FIGURE_DIR, show=False)
summary = create_visualization_summary()
summary_path = REPORT_DIR / "ch07_visualization_summary.md"
summary_path.write_text(
    build_visualization_report(summary),
    encoding="utf-8",
)

# 공통 그래프 생성 결과 중 대표 3개를 제출 Evidence 폴더에도 복사합니다.
for source_name, evidence_name in [
    ("ch07_category_completed_amount_bar.png", "graph01.png"),
    ("ch07_monthly_completed_amount_line.png", "graph02.png"),
    ("ch07_price_completed_quantity_scatter.png", "graph03.png"),
]:
    shutil.copy2(FIGURE_DIR / source_name, IMAGE_DIR / evidence_name)

print("생성 파일")
for path in saved_figures:
    print("-", path)
print("Evidence 이미지:", IMAGE_DIR)
print("요약 보고서:", summary_path)
summary

## 11. Chapter 07 최종 판단

### 가장 중요한 그래프 1개와 이유
월별 완료 주문 금액 선그래프를 가장 중요하게 보았습니다. 시간에 따른 금액 차이와 데이터가 관측되지 않은 기간을 함께 확인할 수 있기 때문입니다.

### 그래프에서 직접 확인한 사실
유효 주문일 기준 월별 금액 합계는 139,437,000원입니다. 관측 월 중 2026년 8월이 16,980,000원으로 가장 높고, 다음 관측 월은 2027년 1월 909,000원입니다. 날짜 결측 완료 상세 6행·1,903,000원은 월별 집계에서 제외했습니다.

### 내가 해석한 내용
월별 금액은 일정하지 않습니다. 2026년 9~12월은 집계 행이 없으므로 그 기간을 0원으로 해석하지 않고, 선그래프에도 공백으로 표시합니다.

### 그래프만으로 말할 수 없는 것
집계 공백이 실제 주문이 없었던 기간인지 자료 누락인지는 알 수 없습니다. 월별 차이의 원인을 프로모션·계절성·가격·재고·고객 행동으로 단정할 수 없으며, 금액만으로 이익이나 충성도를 판단할 수 없습니다.

### 오해를 줄이기 위해 수정하거나 확인한 요소
- 축: 카테고리·월·상품 가격·판매 수량 등 비교 변수와 금액 축을 명시합니다.
- 단위: 금액은 원, 수량은 개, 주문 수는 고유 주문 건입니다.
- 정렬: 카테고리 막대는 금액 내림차순, 월별 자료는 시간순입니다.
- 범위: 완료 주문을 기준으로 합니다. 월별 공백은 0으로 채우지 않고, 상품 FK 미매칭 16행은 카테고리의 `상품 정보 없음`에 보존하며 상품 산점도에서만 제외합니다.
- 개인정보: 고객 그래프와 출력 표는 순위 별칭과 집계 지표만 보이며 고객 ID·이름은 표시하지 않습니다.

### 다음 분석에서 확인하고 싶은 질문
2026년 9~12월 데이터 수집 여부와 가격 최댓값의 원문을 확인하고, 월별 주문 수·평균 주문 금액 및 표준화가 확인된 카테고리별 금액을 비교하겠습니다.


### LLM 시각화 검토 기록
- 목적: 질문에 맞는 그래프 선택, 집계값과 그래프의 일치, 해석 범위와 개인정보 노출 여부를 점검했습니다.
- Safe Context: 고객 이름·연락처·고객 ID는 답안에 표시하지 않고, 집계 금액과 FK·날짜 오류 건수만 사용했습니다.
- Prompt 요약: 완료 주문 집계의 그래프별 관찰·해석·한계와 누락값 처리, 축·단위·시간 공백·개인정보 위험을 실제 집계와 대조하도록 요청했습니다.
- 제안 및 반영: 주문 FK 누락은 주문 분석에서 제외하고, 상품 FK 누락 금액은 카테고리의 미상 범주에 보존하며 산점도에서만 제외했습니다. 원본 CSV 덮어쓰기와 고객 ID 출력을 없애고, 관측이 없는 월은 선그래프에서 공백으로 표시하도록 구성했습니다.
- 사람이 확인한 근거: 첨부 clean CSV의 PK·FK 관계와 `line_total`, 카테고리·월별·상품·고객 집계를 대조했습니다. 시각적 축·범례·그래프 실행 결과는 Codespaces에서 노트북을 실행한 뒤 확인해야 합니다.

---

## 완료 체크

- [ ] 질문을 그래프보다 먼저 정했습니다.
- [ ] 금액성 그래프에 `completed` 주문 범위를 적용했습니다.
- [ ] 집계 검증표의 `passed`가 모두 참인지 실행 후 확인했습니다.
- [ ] 대표 그래프 3개 이상을 재생성했습니다.
- [ ] 그래프와 원본 집계값을 교차 확인했습니다.
- [ ] 축·단위·정렬·범위를 실행 결과에서 확인했습니다.
- [ ] 관계를 원인으로 단정하지 않았습니다.
- [ ] 고객 그래프에서 개인정보를 직접 노출하지 않았습니다.
- [ ] 그래프별 관찰·해석·한계·다음 질문을 작성했습니다.
- [ ] `reports/figures`, `images/graph01.png`~`graph03.png`, 요약 보고서를 재생성했습니다.

다음 Chapter 08에서는 전처리, EDA, 시각화와 보고서를 하나의 작은 분석 프로젝트로 연결합니다.